<a href="https://colab.research.google.com/github/pratapsingh23/countries_state_analysis/blob/main/notebooks/HistGradientBoostClassifer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
# importting dataset
import pandas as pd
from data_loading import data_loading,files
df = data_loading(files['final_df'])


In [31]:
# importing libraries:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support


In [35]:
df.shape

(513801, 52)

In [4]:
df.columns

Index(['collision_index', 'collision_severity', 'date', 'day_of_week', 'time',
       'latitude', 'longitude', 'urban_or_rural_area', 'speed_limit',
       'road_type', 'light_conditions', 'weather_conditions',
       'road_surface_conditions', 'special_conditions_at_site',
       'carriageway_hazards', 'junction_detail', 'junction_control',
       'trunk_road_flag', 'number_of_vehicles', 'number_of_casualties',
       'did_police_officer_attend_scene_of_accident',
       'collision_severity_label', 'day_of_week_label',
       'urban_or_rural_area_label', 'road_type_label',
       'light_conditions_label', 'weather_conditions_label',
       'road_surface_conditions_label', 'special_conditions_at_site_label',
       'carriageway_hazards_label', 'junction_detail_label',
       'junction_control_label', 'trunk_road_flag_label',
       'did_police_officer_attend_scene_of_accident_label', 'min_driver_age',
       'max_driver_age', 'max_vehicle_age', 'max_engine_cc',
       'male_driver_shar

In [6]:
y = df['collision_severity'].isin([1, 2]).astype(int)

In [9]:
num_cols = [
    'speed_limit', 'number_of_vehicles', 'min_driver_age', 'max_driver_age','max_vehicle_age', 'max_engine_cc', 'male_driver_share', 'avg_casualty_imd'
]

bin_cols = [
    'has_cycle', 'has_motorcycle', 'has_goods_vehicle', 'has_bus', 'has_vulnerable_vehicle', 'has_overtaking', 'has_turning', 'has_vulnerable_casualty', 'has_escooter_casualty', 'has_pedestrian_movement', 'trunk_road_flag'
]

cat_cols = [
    'urban_or_rural_area', 'road_type', 'light_conditions', 'weather_conditions', 'road_surface_conditions',
    'special_conditions_at_site', 'carriageway_hazards','junction_detail', 'junction_control', 'day_of_week', 'hour'
]

df[cat_cols] = df[cat_cols].astype('category')

In [12]:
X = df[num_cols + bin_cols + cat_cols]

In [13]:
# train test split:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("\ny_train class counts:")
print(y_train.value_counts())
print("\ny_test class counts:")
print(y_test.value_counts())

X_train (411040, 30)
X_test (102761, 30)

y_train class counts:
collision_severity
0    311547
1     99493
Name: count, dtype: int64

y_test class counts:
collision_severity
0    77888
1    24873
Name: count, dtype: int64


In [21]:
pipe = Pipeline(steps=[
    ('hgbc', HistGradientBoostingClassifier(
        categorical_features='from_dtype',
        class_weight='balanced',
        random_state=42,
        learning_rate=0.05,
        max_iter=400,
        max_leaf_nodes=60,
        max_depth=None,
        min_samples_leaf=20,
        l2_regularization=0.0,
        early_stopping=True,
        validation_fraction=0.1
    ))
])

pipe.fit(X_train, y_train)


Pipeline(steps=[('hgbc',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                early_stopping=True,
                                                learning_rate=0.05,
                                                max_iter=400, max_leaf_nodes=60,
                                                random_state=42))])

In [22]:
y_pred = pipe.predict(X_test)


print(classification_report(y_test, y_pred, target_names=['Slight (0)', 'KSI (1)']))

print("confusion matrix: \n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

  Slight (0)       0.86      0.64      0.73     77888
     KSI (1)       0.37      0.67      0.48     24873

    accuracy                           0.64    102761
   macro avg       0.61      0.65      0.60    102761
weighted avg       0.74      0.64      0.67    102761

confusion matrix: 
 [[49496 28392]
 [ 8142 16731]]


In [25]:
# best threshold finding:
y_proba = pipe.predict_proba(X_test)[:, 1]
best = (0, 0, 0, 0)
print("thr | precision | recall | f1")
print("-" * 30)

for t in np.arange(0.20, 0.85, 0.05):
  y_pred_t = (y_proba >= t).astype(int)
  p, r, f1, _ = precision_recall_fscore_support(
      y_test, y_pred_t, average = 'binary', pos_label=1
  )
  print(f"{t:.2f} | {p:.3f} | {r:.3f} | {f1:.3f}")
  if f1 > best[3]:
    best = [t, p, r, f1]


print("\nBest threshold:", round(best[0], 2),
      "-> P:", round(best[1], 3), "F1:", round(best[3], 3))

thr | precision | recall | f1
------------------------------
0.20 | 0.262 | 0.985 | 0.414
0.25 | 0.273 | 0.965 | 0.426
0.30 | 0.287 | 0.935 | 0.440
0.35 | 0.303 | 0.891 | 0.453
0.40 | 0.323 | 0.837 | 0.466
0.45 | 0.344 | 0.762 | 0.474
0.50 | 0.371 | 0.673 | 0.478
0.55 | 0.404 | 0.563 | 0.470
0.60 | 0.446 | 0.439 | 0.442
0.65 | 0.494 | 0.314 | 0.384
0.70 | 0.547 | 0.209 | 0.303
0.75 | 0.612 | 0.132 | 0.218
0.80 | 0.673 | 0.072 | 0.130

Best threshold: 0.5 -> P: 0.371 F1: 0.478


In [26]:
pipe2 = Pipeline(steps=[
    ('hgbc', HistGradientBoostingClassifier(
        categorical_features='from_dtype',
        class_weight='balanced',
        random_state=42,
        learning_rate=0.05,
        max_iter=600,
        max_leaf_nodes=255,
        max_depth=None,
        min_samples_leaf=100,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1
    ))
])

pipe2.fit(X_train, y_train)

Pipeline(steps=[('hgbc',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                early_stopping=True,
                                                l2_regularization=1.0,
                                                learning_rate=0.05,
                                                max_iter=600,
                                                max_leaf_nodes=255,
                                                min_samples_leaf=100,
                                                random_state=42))])

In [30]:
y_pred2 = pipe2.predict(X_test)

y_train_pred = pipe2.predict(X_train)
print(classification_report(y_test, y_pred2, target_names=['Slight (0)', 'KSI (1)']))

print("confusion matrix: \n", confusion_matrix(y_test, y_pred2))




              precision    recall  f1-score   support

  Slight (0)       0.86      0.65      0.74     77888
     KSI (1)       0.37      0.66      0.48     24873

    accuracy                           0.65    102761
   macro avg       0.62      0.65      0.61    102761
weighted avg       0.74      0.65      0.67    102761

confusion matrix: 
 [[50352 27536]
 [ 8374 16499]]


